# 🔵 Notebook 05 — K-Means: Segmentação de Padrões de Incidentes
## Predictfy × Locaweb — FIAP Challenge 2026

---

**Objetivo:** Descobrir **grupos naturais** (clusters) nos incidentes KPI que compartilham padrões
de horário, prioridade, volume e contexto operacional — sem usar o target (aprendizado não-supervisionado).

| | |
|---|---|
| **Input**  | `data/processed/incidents_features.parquet` (gerado pelo Notebook 02) |
| **Output** | `outputs/clusters.json` |
| **Algoritmo** | K-Means com normalização StandardScaler + PCA para visualização |
| **K mínimo** | 4 — K=2 e K=3 são divisões triviais sem utilidade operacional |

---

In [ ]:
# ── 0. Setup ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
import json, os
from datetime import date

pd.set_option('display.max_columns', 30)
np.random.seed(42)

print('✅ Imports ok')


## 1. Carregar Dataset de Features

In [ ]:
# ── Caminhos ──────────────────────────────────────────────────────────────────
PARQUET_PATH = '../data/processed/incidents_features.parquet'
RAW_PATH     = '../data/raw/LW-DATASET.xlsx'

# ── Tentar carregar Parquet gerado pelo Notebook 02 ───────────────────────────
if os.path.exists(PARQUET_PATH):
    df = pd.read_parquet(PARQUET_PATH)
    print(f'✅ Parquet carregado: {df.shape}')
    print(f'   Colunas: {list(df.columns)}')

else:
    print('⚠️  incidents_features.parquet não encontrado.')
    print('   Gerando features inline a partir do dataset bruto...')
    print('   ⚡ Para resultado idêntico ao Notebook 02, execute-o primeiro.\n')

    df_raw = pd.read_excel(RAW_PATH, parse_dates=['Aberto', 'Resolvido', 'Encerrado'])
    print(f'   Dataset bruto: {df_raw.shape}')

    kpi = df_raw[df_raw['Entrou para KPI?'] == 'SIM'].copy()
    kpi = kpi[kpi['Prioridade'].isin(['2 - Alta', '3 - Média'])].copy()
    kpi = kpi.sort_values('Aberto').reset_index(drop=True)
    print(f'   Subset KPI: {kpi.shape}')

    OLA_MAP = {'2 - Alta': 14400, '3 - Média': 43200}
    kpi['ola_segundos'] = kpi['Prioridade'].map(OLA_MAP)
    kpi['target_ola']   = (kpi['Duração'] > kpi['ola_segundos']).astype(int)

    kpi['data']       = kpi['Aberto'].dt.normalize()
    kpi['hora']       = kpi['Aberto'].dt.hour
    kpi['dia_semana'] = kpi['Aberto'].dt.dayofweek
    kpi['mes']        = kpi['Aberto'].dt.month
    kpi['trimestre']  = kpi['Aberto'].dt.quarter
    kpi['dia_mes']    = kpi['Aberto'].dt.day
    kpi['semana_ano'] = kpi['Aberto'].dt.isocalendar().week.astype(int)

    kpi['is_horario_comercial'] = (
        (kpi['hora'] >= 8) & (kpi['hora'] <= 18) & (kpi['dia_semana'] < 5)
    ).astype(int)
    kpi['is_fim_de_semana'] = (kpi['dia_semana'] >= 5).astype(int)
    kpi['is_segunda_terca'] = (kpi['dia_semana'] <= 1).astype(int)

    def periodo_dia(h):
        if 0 <= h < 6:    return 0
        elif 6 <= h < 12: return 1
        elif 12 <= h < 18: return 2
        else:             return 3
    kpi['periodo_dia'] = kpi['hora'].apply(periodo_dia)

    vol = kpi.groupby('data').size().reset_index(name='vol_dia').sort_values('data')
    vol['lag_1d']      = vol['vol_dia'].shift(1)
    vol['lag_7d']      = vol['vol_dia'].shift(7)
    vol['rolling_7d']  = vol['vol_dia'].rolling(7,  min_periods=1).mean().round(2)
    vol['rolling_30d'] = vol['vol_dia'].rolling(30, min_periods=1).mean().round(2)
    kpi = kpi.merge(vol[['data','lag_1d','lag_7d','rolling_7d','rolling_30d']], on='data', how='left')

    kpi['prioridade_bin'] = (kpi['Prioridade'] == '2 - Alta').astype(int)

    for col in ['Produto', 'Categoria', 'Subcategoria', 'Grupo designado', 'Aberto por']:
        kpi[col] = kpi[col].fillna('DESCONHECIDO')

    le = LabelEncoder()
    kpi['produto_enc']      = le.fit_transform(kpi['Produto'])
    kpi['categoria_enc']    = le.fit_transform(kpi['Categoria'])
    kpi['subcategoria_enc'] = le.fit_transform(kpi['Subcategoria'])
    kpi['grupo_enc']        = le.fit_transform(kpi['Grupo designado'])
    kpi['aberto_por_enc']   = le.fit_transform(kpi['Aberto por'])

    kpi['produto_freq'] = kpi['Produto'].map(
        kpi['Produto'].value_counts(normalize=True)).round(6)
    kpi['grupo_freq']   = kpi['Grupo designado'].map(
        kpi['Grupo designado'].value_counts(normalize=True)).round(6)

    kpi['mes_sin'] = np.sin(2 * np.pi * kpi['mes'] / 12)
    kpi['mes_cos'] = np.cos(2 * np.pi * kpi['mes'] / 12)

    df = kpi.copy()
    print(f'\n✅ Features geradas inline: {df.shape}')

TARGET = 'target_ola'
print(f'\n📊 Dataset carregado: {len(df):,} incidentes KPI')
print(f'   Violações: {df[TARGET].sum():,} ({df[TARGET].mean()*100:.2f}%)')


## 2. Selecionar Features para Clustering

In [ ]:
# ── Features para K-Means ─────────────────────────────────────────────────────
# K-Means usa distância euclidiana → features devem ser contínuas ou ordinais com sentido
# NÃO usar label encoding de categóricas nominais (implica ordinalidade inexistente)
# Usar frequency encoding (captura importância relativa sem implicar ordem)

CLUSTER_FEATURES = [
    # Temporais — quando o incidente foi aberto
    'hora', 'dia_semana', 'mes', 'periodo_dia',
    # Contexto operacional
    'is_horario_comercial', 'is_fim_de_semana', 'prioridade_bin',
    # Volume histórico (carga no dia da abertura)
    'lag_1d', 'rolling_7d',
    # Frequência de produto/grupo (captura importância sem assumir ordem)
    'produto_freq', 'grupo_freq',
]

print(f'📊 Features selecionadas para clustering: {len(CLUSTER_FEATURES)}')
for f in CLUSTER_FEATURES:
    print(f'   • {f}')

df_cl = df[CLUSTER_FEATURES + [TARGET]].copy()
df_cl[['lag_1d','rolling_7d']] = df_cl[['lag_1d','rolling_7d']].fillna(0)

assert df_cl.isnull().sum().sum() == 0, '❌ Nulos encontrados!'
print(f'\n✅ Dataset de clustering: {df_cl.shape}  |  0 nulos')


## 3. Normalização — StandardScaler

In [ ]:
# ── Normalizar com StandardScaler ────────────────────────────────────────────
scaler   = StandardScaler()
X_raw    = df_cl[CLUSTER_FEATURES].values
X_scaled = scaler.fit_transform(X_raw)

print('✅ Normalização aplicada (StandardScaler)')
print(f'   Shape: {X_scaled.shape}')

fig, axes = plt.subplots(2, len(CLUSTER_FEATURES), figsize=(22, 5))
for i, feat in enumerate(CLUSTER_FEATURES):
    axes[0, i].hist(X_raw[:, i], bins=30, color='#ff9f0a', alpha=0.8, edgecolor='white')
    axes[0, i].set_title(feat, fontsize=7); axes[0, i].tick_params(labelsize=6)
    axes[1, i].hist(X_scaled[:, i], bins=30, color='#1a73e8', alpha=0.8, edgecolor='white')
    axes[1, i].tick_params(labelsize=6)

axes[0, 0].set_ylabel('Original', fontsize=9)
axes[1, 0].set_ylabel('Normalizado', fontsize=9)
plt.suptitle('Distribuições — Antes vs Depois do StandardScaler', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()


## 4. Determinar K Ideal — Elbow + Silhouette + CH + Davies-Bouldin

> **Por que K=2 não serve?**
> Quando o dataset contém features binárias dominantes (`is_horario_comercial`,
> `is_fim_de_semana`, `prioridade_bin`), o K-Means tende a separar o espaço em exatamente
> **2 grupos** — aqueles com/sem essas flags ativas. Isso é matematicamente correto, mas
> operacionalmente inútil: não gera segmentos acionáveis para as equipes de suporte.
>
> **Por que K ímpar?**
> Em dados de incidentes ITSM, K ímpar produz clusters mais estáveis: evita que o algoritmo
> fique "preso" em partições simétricas de 2×N que naturalmente emergem de features binárias.
> Com K ímpar, um cluster central "absorve" os casos limítrofes, resultando em fronteiras
> mais nítidas e perfis mais distintos.
>
> **K_MIN = 5** (ímpar): exige ao menos 5 clusters para segmentação operacional útil.
> Candidatos válidos: K ∈ {5, 7} — seleciona o melhor por silhouette.
>
> **Três métricas complementares:**
> - **Silhouette** (↑ melhor): coesão vs separação, sensível a clusters de tamanhos diferentes
> - **Calinski-Harabasz** (↑ melhor): razão dispersão inter-cluster / intra-cluster
> - **Davies-Bouldin** (↓ melhor): média das razões de similaridade entre pares de clusters

In [ ]:
# ── Avaliar K de 2 a 10 com 3 métricas ──────────────────────────────────────
# K_MIN = 5 (ímpar): K par tende a criar partições simétricas artificiais em dados ITSM
# K ímpar produz um cluster "central" que absorve casos limítrofes → fronteiras mais nítidas
# Candidatos válidos para seleção: K ∈ {5, 7} (ímpares entre K_MIN e K_MAX_UTIL)

K_MIN      = 5   # mínimo ímpar com utilidade operacional
K_MAX_UTIL = 9   # limite superior (ímpar)
K_RANGE    = range(2, 11)

N_SAMPLE   = min(10_000, len(X_scaled))
idx_sample = np.random.choice(len(X_scaled), N_SAMPLE, replace=False)
X_sample   = X_scaled[idx_sample]

inertias   = []
sil_scores = []
ch_scores  = []
db_scores  = []

print('🔍 Avaliando K de 2 a 10 (3 métricas)...')
print(f'   Candidatos válidos (ímpar, K_MIN={K_MIN}..{K_MAX_UTIL}): {[k for k in K_RANGE if K_MIN <= k <= K_MAX_UTIL and k % 2 == 1]}')
print(f'   (Silhouette calculado em amostra de {N_SAMPLE:,} pontos)\n')
print(f'   {"K":>3}  {"Inertia":>12}  {"Silhouette":>12}  {"Calinski-H":>12}  {"Davies-B":>10}')
print('   ' + '-'*56)

for k in K_RANGE:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10, max_iter=300)
    lbl = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)

    lbl_sample = lbl[idx_sample]
    sil = silhouette_score(X_sample, lbl_sample, sample_size=min(5000, N_SAMPLE))
    ch  = calinski_harabasz_score(X_scaled, lbl)
    db  = davies_bouldin_score(X_scaled, lbl)

    sil_scores.append(sil)
    ch_scores.append(ch)
    db_scores.append(db)

    is_candidato = K_MIN <= k <= K_MAX_UTIL and k % 2 == 1
    flag = ' ◀ candidato' if is_candidato else (' (par — excluído)' if k % 2 == 0 and K_MIN <= k <= K_MAX_UTIL else '')
    print(f'   {k:>3}  {km.inertia_:>12,.1f}  {sil:>12.4f}  {ch:>12.1f}  {db:>10.4f}{flag}')

# ── Selecionar K: apenas ímpares dentro de [K_MIN, K_MAX_UTIL] ───────────────
valid = [(k, s, c, d) for k, s, c, d in zip(K_RANGE, sil_scores, ch_scores, db_scores)
         if K_MIN <= k <= K_MAX_UTIL and k % 2 == 1]

if not valid:
    raise ValueError(f'Nenhum K ímpar válido encontrado em [{K_MIN}, {K_MAX_UTIL}]')

best_k_sil = max(valid, key=lambda x: x[1])[0]   # maior silhouette
best_k_ch  = max(valid, key=lambda x: x[2])[0]   # maior Calinski-Harabasz
best_k_db  = min(valid, key=lambda x: x[3])[0]   # menor Davies-Bouldin

print()
print(f'🎯 K ótimo por critério (ímpares em [{K_MIN}, {K_MAX_UTIL}]):')
print(f'   Silhouette (↑):        K = {best_k_sil}')
print(f'   Calinski-Harabasz (↑): K = {best_k_ch}')
print(f'   Davies-Bouldin (↓):    K = {best_k_db}')

# Voto majoritário — empate desempata por silhouette
from collections import Counter
votos      = [best_k_sil, best_k_ch, best_k_db]
k_vencedor = Counter(votos).most_common(1)[0][0]
if Counter(votos).most_common(1)[0][1] == 1:
    k_vencedor = best_k_sil

print(f'\n✅ K selecionado (ímpar, voto majoritário): K = {k_vencedor}')

# ── Visualização ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ks        = list(K_RANGE)
impares   = [k for k in ks if k % 2 == 1 and K_MIN <= k <= K_MAX_UTIL]

for ax, vals, ylabel, titulo, cor in [
    (axes[0], inertias,   'Inertia (WCSS)',         'Elbow Method',       '#1a73e8'),
    (axes[1], sil_scores, 'Silhouette Score',        'Silhouette por K',   '#30d158'),
    (axes[2], db_scores,  'Davies-Bouldin (↓ melhor)','Davies-Bouldin por K','#ff9f0a'),
]:
    ax.plot(ks, vals, color=cor, lw=2, marker='o', ms=5)
    for k_imp in impares:
        ax.axvline(k_imp, color='#d3d3d3', ls=':', lw=1)
    ax.axvline(k_vencedor, color='#ff2d55', ls='--', lw=2,
               label=f'K={k_vencedor} selecionado')
    ax.set_xlabel('K'); ax.set_ylabel(ylabel)
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
    # Marcar K_MIN
    ax.axvline(K_MIN, color='#ff9f0a', ls='--', lw=1, alpha=0.7, label=f'K_MIN={K_MIN}')

plt.suptitle(f'Determinação do K Ideal — apenas K ímpar ≥ {K_MIN}', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print()
print(f'⚠️  Candidatos avaliados (ímpares): {[v[0] for v in valid]}')
print(f'   Silhouette por candidato: {[(v[0], round(v[1],4)) for v in valid]}')
print(f'   K_MIN={K_MIN} (ímpar) evita partições simétricas artificiais de K par em dados ITSM.')

## 5. Treinar K-Means com K Selecionado

In [ ]:
# ── Treinar K-Means com K selecionado ─────────────────────────────────────────
K_FINAL = k_vencedor   # resultado do voto majoritário (K_MIN ≤ K_FINAL ≤ 8)
# Para forçar manualmente: K_FINAL = 4

print(f'🔵 Treinando K-Means com K={K_FINAL}...')

kmeans   = KMeans(n_clusters=K_FINAL, random_state=42, n_init=20, max_iter=500)
labels   = kmeans.fit_predict(X_scaled)
df_cl    = df_cl.copy()
df_cl['cluster'] = labels

inertia_final = kmeans.inertia_
sil_final     = silhouette_score(X_scaled, labels, sample_size=min(10000, len(X_scaled)))
ch_final      = calinski_harabasz_score(X_scaled, labels)
db_final      = davies_bouldin_score(X_scaled, labels)

print(f'✅ K-Means treinado!')
print(f'   K              = {K_FINAL}')
print(f'   Inertia        = {inertia_final:,.1f}')
print(f'   Silhouette     = {sil_final:.4f}')
print(f'   Calinski-H     = {ch_final:.1f}')
print(f'   Davies-Bouldin = {db_final:.4f}')
print()

sizes = df_cl['cluster'].value_counts().sort_index()
print('📊 Distribuição dos clusters:')
for c, n in sizes.items():
    viol = df_cl[df_cl['cluster']==c][TARGET].sum()
    print(f'   Cluster {c}: {n:>6,} incidentes  ({n/len(df_cl)*100:.1f}%)  |  {viol} violações  ({viol/n*100:.2f}%)')


## 6. Perfil de Cada Cluster

In [ ]:
# ── Calcular perfil estatístico de cada cluster ────────────────────────────────
perfil_raw = df_cl.groupby('cluster').agg(
    {**{f: 'mean' for f in CLUSTER_FEATURES},
     TARGET: ['mean', 'sum', 'count']}
)
perfil_raw.columns = [*CLUSTER_FEATURES, 'taxa_violacao', 'total_violacoes', 'total_incidentes']
perfil_raw = perfil_raw.reset_index()

print('📊 Perfil médio por cluster:')
print(perfil_raw[['cluster','total_incidentes','taxa_violacao',
                   'hora','dia_semana','prioridade_bin',
                   'is_horario_comercial','is_fim_de_semana']].to_string(index=False))

perfil_norm = perfil_raw[CLUSTER_FEATURES].copy()
perfil_norm = (perfil_norm - perfil_norm.min()) / (perfil_norm.max() - perfil_norm.min() + 1e-9)
perfil_norm.index = [f'Cluster {c}' for c in perfil_raw['cluster']]

fig, ax = plt.subplots(figsize=(14, max(3, K_FINAL + 1)))
sns.heatmap(perfil_norm, annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, ax=ax)
ax.set_title('Perfil Normalizado dos Clusters (0=mín · 1=máx)', fontsize=12, fontweight='bold')
ax.set_xlabel('Feature', fontsize=10)
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()


## 7. Visualização 2D — PCA

In [ ]:
pca   = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f'📊 PCA — variância explicada:')
print(f'   PC1: {pca.explained_variance_ratio_[0]*100:.1f}%')
print(f'   PC2: {pca.explained_variance_ratio_[1]*100:.1f}%')
print(f'   Total: {pca.explained_variance_ratio_.sum()*100:.1f}%')

CORES = ['#1a73e8','#ff9f0a','#30d158','#ff2d55','#af52de',
         '#5ac8fa','#ff6b35','#34c759']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for c in sorted(df_cl['cluster'].unique()):
    mask = df_cl['cluster'] == c
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    c=CORES[c % len(CORES)], alpha=0.3, s=5,
                    label=f'Cluster {c} (n={mask.sum():,})')
axes[0].set_title('Clusters — PCA 2D', fontsize=12, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(fontsize=9, markerscale=3); axes[0].grid(alpha=0.2)

cor_viol = df_cl[TARGET].map({0: '#5ac8fa', 1: '#ff2d55'}).values
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=cor_viol, alpha=0.3, s=5)
axes[1].set_title('Violações OLA — PCA 2D (vermelho=SIM)', fontsize=12, fontweight='bold')
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].grid(alpha=0.2)

plt.suptitle('Espaço PCA — K-Means Clusters', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 8. Análise de Violações por Cluster

In [ ]:
PERIODO_MAP = {0: 'madrugada', 1: 'manhã', 2: 'tarde', 3: 'noite'}

print('📊 Análise de violações por cluster:\n')
cluster_info = []

for c in sorted(df_cl['cluster'].unique()):
    sub  = df_cl[df_cl['cluster'] == c]
    n    = len(sub)
    viol = int(sub[TARGET].sum())
    taxa = viol / n * 100

    hora_med   = sub['hora'].mean()
    dow_med    = sub['dia_semana'].mean()
    pct_p2     = sub['prioridade_bin'].mean() * 100
    pct_comerc = sub['is_horario_comercial'].mean() * 100
    pct_fds    = sub['is_fim_de_semana'].mean() * 100
    periodo_m  = PERIODO_MAP[int(sub['periodo_dia'].mode()[0])]

    if pct_p2 > 40:
        nome = f'Alta Prioridade P2-dominante'
    elif pct_fds > 30:
        nome = f'Fim de Semana / Noturno'
    elif hora_med < 8 or hora_med > 18:
        nome = f'Fora do Horário Comercial'
    elif taxa > perfil_raw['taxa_violacao'].mean() * 1.3:
        nome = f'Risco Elevado — {periodo_m.capitalize()}'
    else:
        nome = f'Operação Normal — {periodo_m.capitalize()}'

    cluster_info.append({
        'id': int(c), 'nome': nome,
        'tamanho': n, 'pct_total': round(n/len(df_cl)*100, 2),
        'taxa_violacao_pct': round(taxa, 3),
        'total_violacoes': viol,
        'perfil': {
            'hora_media':               round(hora_med, 1),
            'dia_semana_medio':         round(dow_med, 1),
            'prioridade_p2_pct':        round(pct_p2, 1),
            'horario_comercial_pct':    round(pct_comerc, 1),
            'fim_de_semana_pct':        round(pct_fds, 1),
            'periodo_dia_predominante': periodo_m,
        }
    })

    print(f'  Cluster {c} — {nome}')
    print(f'    Tamanho:             {n:>7,}  ({n/len(df_cl)*100:.1f}%)')
    print(f'    Taxa de violação:    {taxa:>7.3f}%  ({viol} violações)')
    print(f'    Hora média:          {hora_med:>7.1f}h')
    print(f'    % P2 (Alta):         {pct_p2:>7.1f}%')
    print(f'    % Horário comercial: {pct_comerc:>7.1f}%')
    print(f'    % Fim de semana:     {pct_fds:>7.1f}%')
    print(f'    Período dominante:   {periodo_m}')
    print()

CORES = ['#1a73e8','#ff9f0a','#30d158','#ff2d55','#af52de','#5ac8fa','#ff6b35','#34c759']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ids    = [c['id']               for c in cluster_info]
taxas  = [c['taxa_violacao_pct'] for c in cluster_info]
sizes_ = [c['tamanho']          for c in cluster_info]
cores_ = [CORES[c['id'] % len(CORES)] for c in cluster_info]

bars0 = axes[0].bar([f'C{i}' for i in ids], taxas, color=cores_, edgecolor='white')
axes[0].set_title('Taxa de Violação por Cluster (%)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('% Violações')
for bar, v in zip(bars0, taxas):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.02,
                 f'{v:.2f}%', ha='center', va='bottom', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

bars1 = axes[1].bar([f'C{i}' for i in ids], sizes_, color=cores_, edgecolor='white')
axes[1].set_title('Tamanho dos Clusters', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Nº de incidentes')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
for bar, v in zip(bars1, sizes_):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 50,
                 f'{v:,}', ha='center', va='bottom', fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle(f'Análise dos {K_FINAL} Clusters K-Means', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 9. Exportar `outputs/clusters.json`

In [ ]:
output = {
    'modelo':    'kmeans_segmentacao',
    'gerado_em': date.today().strftime('%Y-%m-%d'),
    'versao':    'v2',
    'k':         K_FINAL,
    'k_min_utilizado': K_MIN,
    'metricas': {
        'inertia':           round(float(inertia_final), 2),
        'silhouette_score':  round(float(sil_final), 4),
        'calinski_harabasz': round(float(ch_final), 2),
        'davies_bouldin':    round(float(db_final), 4),
        'features_usadas':   len(CLUSTER_FEATURES),
        'total_incidentes':  int(len(df_cl)),
        'pca_variancia_2d':  round(float(pca.explained_variance_ratio_.sum()), 4),
    },
    'features_clustering': CLUSTER_FEATURES,
    'clusters': cluster_info,
}

os.makedirs('../outputs', exist_ok=True)
with open('../outputs/clusters.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print('✅ outputs/clusters.json exportado! (v2)')
print(f'\n📊 Resumo:')
print(f'   K              = {K_FINAL}  (K_MIN={K_MIN})')
print(f'   Silhouette     = {sil_final:.4f}')
print(f'   Calinski-H     = {ch_final:.1f}')
print(f'   Davies-Bouldin = {db_final:.4f}')
print(f'   Variância PCA  = {pca.explained_variance_ratio_.sum()*100:.1f}%')
for c in cluster_info:
    print(f'   Cluster {c["id"]}: {c["tamanho"]:>6,} incidentes  |  {c["taxa_violacao_pct"]:.2f}% violações  → {c["nome"]}')


## ✅ Notebook 05 — Concluído (v2)

### Resumo
K-Means aplicado com K_MIN=4 para garantir segmentação operacionalmente útil.
Três métricas de avaliação (Silhouette, Calinski-Harabasz, Davies-Bouldin) votam no K ideal.
Cada cluster é perfilado e nomeado semanticamente com base em suas características dominantes.

### Por que K_MIN=4?
K=2 sempre vence a silhouette puro quando features binárias dominam a distância euclidiana.
O resultado seria uma divisão trivial (horário comercial vs fora), sem valor para as equipes de suporte.

### Output gerado
- `outputs/clusters.json` (v2) — K, 3 métricas, perfil e taxa de violação por cluster, k_min registrado

### Próximo passo
- **Notebook 06** — SHAP: Explicabilidade aprofundada do XGBoost
- **Notebook 07** — Projeção anual de KPI atingimento (já concluído)